In [ ]:
import pandas as pd
import plotly.graph_objects as go
import re

# --- 1. CONFIGURATION ---
input_file = "/data484_4/txia2/gwas_practice/drug_GREP/GREP/output/mocov2_partial_fit_5std.ICD.txt"

# Filter Threshold
P_VALUE_THRESHOLD = 0.05

# Visual Settings
FONT_SIZE = 16
OPACITY = 0.4  # Light pastel colors

# Colors (ICD Chapters) - Rotated so red colors go to F and G
chapter_colors = {
    'A': f'rgba(255, 195, 0, {OPACITY})',   # Yellow (was E)
    'B': f'rgba(218, 247, 166, {OPACITY})', # Light Green (was F)
    'C': f'rgba(51, 255, 87, {OPACITY})',   # Green (was G)
    'D': f'rgba(51, 255, 189, {OPACITY})',  # Teal (was H)
    'E': f'rgba(51, 128, 255, {OPACITY})',  # Blue (was I)
    'F': f'rgba(199, 0, 57, {OPACITY})',    # Red (was B) - RED!
    'G': f'rgba(255, 87, 51, {OPACITY})',   # Orange (was A) - RED!
    'H': f'rgba(93, 51, 255, {OPACITY})',   # Indigo (was J)
    'I': f'rgba(131, 51, 255, {OPACITY})',  # Violet (was K)
    'J': f'rgba(51, 128, 255, {OPACITY})',  # Blue
    'K': f'rgba(255, 51, 246, {OPACITY})',  # Pink (was M)
    'L': f'rgba(255, 51, 134, {OPACITY})',  # Hot Pink (was N)
    'M': f'rgba(144, 12, 63, {OPACITY})',   # Purple/Red (was C)
    'N': f'rgba(88, 24, 69, {OPACITY})',    # Dark Purple (was D)
}
fallback_color = f'rgba(136, 136, 136, {OPACITY})'

# Legend Names
icd_legend_map = {
    'A': 'Infectious (A)', 'B': 'Infectious (B)', 'C': 'Neoplasms',
    'D': 'Blood/Immune', 'E': 'Endocrine', 'F': 'Mental',
    'G': 'Nervous System', 'H': 'Eye/Ear', 'I': 'Circulatory',
    'J': 'Respiratory', 'K': 'Digestive', 'L': 'Skin',
    'M': 'Musculoskeletal', 'N': 'Genitourinary'
}

# --- 2. LOAD AND FILTER DATA ---
df = pd.read_csv(input_file, sep="\t")
df.columns = [c.lstrip("#") for c in df.columns]

# Clean P-Value column: Coerce errors to NaN (handles garbage text headers)
df['FisherExactP'] = pd.to_numeric(df['FisherExactP'], errors='coerce')

# *** PRIMARY FILTER: Keep only significant rows ***
df_sig = df[df['FisherExactP'] < P_VALUE_THRESHOLD].copy()

print(f"Original rows: {len(df)}. Rows with P < {P_VALUE_THRESHOLD}: {len(df_sig)}")

# --- 3. PARSE PATHS ---
flat_data = []

def clean_icd(val):
    if pd.isna(val): return None
    return val.split()[0] # Just the code, e.g., "C00-C14"

for _, row in df_sig.iterrows():
    raw_disease = row.get("Group")
    drug_field = row.get("TargetGene:DrugNames")

    if pd.isna(drug_field): continue

    disease_code = clean_icd(raw_disease)
    if not disease_code: continue
    
    # Determine Chapter
    chapter = disease_code[0] if disease_code[0].isalpha() else 'Other'

    # Parse GENE:drug,drug
    for block in str(drug_field).split(";"):
        if ":" not in block: continue
        gene, drugs_str = block.split(":", 1)
        gene = gene.strip()
        
        if not gene: continue

        for drug in drugs_str.split(","):
            drug = drug.strip()
            if drug:
                flat_data.append({
                    'gene': gene, 
                    'drug': drug, 
                    'disease': disease_code, 
                    'chapter': chapter
                })

# DataFrame of all valid connections
df_paths = pd.DataFrame(flat_data)

# Calculate statistics
num_sig_associations = len(df_sig)
num_unique_combinations = df_paths.drop_duplicates(['gene', 'drug', 'disease']).shape[0]

# Calculate unique counts
num_unique_genes = df_paths['gene'].nunique()
num_unique_drugs = df_paths['drug'].nunique()
num_unique_diseases = df_paths['disease'].nunique()

print(f"Number of significant associations: {num_sig_associations}")
print(f"Number of unique drug-gene-disease combinations: {num_unique_combinations}")
print(f"Number of unique genes: {num_unique_genes}")
print(f"Number of unique drugs: {num_unique_drugs}")
print(f"Number of unique diseases: {num_unique_diseases}")
if df_paths.empty:
    print("No data found passing the P-value filter.")
    exit()

# --- 4. PREPARE SANKEY DATA ---
# Drugs to highlight in red
highlight_drugs = {
    'amlodipine', 'asp-3258', 'butalbital', 'cilomilast', 'clobazam', 'clonazepam',
    'compound 1', 'diclofenac', 'dipyridamole', 'lorazepam', 'meprobamate', 'midazolam',
    'nbqx', 'nicardipine', 'nifedipine', 'nitroprusside', 'pentobarbital', 'propofol',
    'prostratin', 'sevoflurane', 'tas-203', 'th-9229', 'topiramate', 'trimebutine',
    'verapamil'
}

# Unique Lists
top_drugs = df_paths['drug'].value_counts().head(30).index
df_paths = df_paths[df_paths['drug'].isin(top_drugs)].copy()

genes = sorted(df_paths['gene'].unique())
drugs = sorted(df_paths['drug'].unique())
diseases = sorted(df_paths['disease'].unique())

all_labels = genes + drugs + diseases
node_map = {label: i for i, label in enumerate(all_labels)}

# Create node colors: red for highlighted drugs, black for others
node_colors = []
for label in all_labels:
    if label.lower() in highlight_drugs:
        node_colors.append('red')
    else:
        node_colors.append('black')

# Create labels with red text for highlighted drugs
formatted_labels = []
for label in all_labels:
    if label.lower() in highlight_drugs:
        formatted_labels.append(f'<span style="color:red">{label}</span>')
    else:
        formatted_labels.append(label)

sources = []
targets = []
values = []
link_colors = []

# Path 1: Gene -> Drug
# Grouping maintains flow volume
gd_group = df_paths.groupby(['gene', 'drug', 'chapter']).size().reset_index(name='flow')
for _, row in gd_group.iterrows():
    sources.append(node_map[row['gene']])
    targets.append(node_map[row['drug']])
    values.append(row['flow'])
    link_colors.append(chapter_colors.get(row['chapter'], fallback_color))

# Path 2: Drug -> Disease
dd_group = df_paths.groupby(['drug', 'disease', 'chapter']).size().reset_index(name='flow')
for _, row in dd_group.iterrows():
    sources.append(node_map[row['drug']])
    targets.append(node_map[row['disease']])
    values.append(row['flow'])
    link_colors.append(chapter_colors.get(row['chapter'], fallback_color))

# --- 5. DYNAMIC HEIGHT ---
# Calculate height based on the busiest column so text doesn't overlap
max_nodes = max(len(genes), len(drugs), len(diseases))
dynamic_height = max(1000, max_nodes * 25) # 25px per node minimum

# --- 6. PLOT ---
fig = go.Figure()

# Sankey Trace
fig.add_trace(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=formatted_labels,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
))

# Legend Trace (Invisible Markers)
present_chapters = sorted(df_paths['chapter'].unique())
for chap in present_chapters:
    color_rgba = chapter_colors.get(chap, fallback_color)
    color_solid = color_rgba.replace(f", {OPACITY})", ", 1.0)")
    label_name = icd_legend_map.get(chap, f"Chapter {chap}")
    
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color=color_solid),
        name=label_name
    ))

# Layout: No Axis, Transparent Background
fig.update_layout(
    title_text=f"Significant Interactions (P < {P_VALUE_THRESHOLD})",
    font=dict(size=FONT_SIZE),
    height=dynamic_height,
    width=1600,
    showlegend=True,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

fig.show()

Original rows: 221. Rows with P < 0.05: 28
Number of significant associations: 28
Number of unique drug-gene-disease combinations: 989
Number of unique genes: 80
Number of unique drugs: 367
Number of unique diseases: 28


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import re

# --- 1. CONFIGURATION ---
input_file = "/data484_4/txia2/gwas_practice/drug_GREP/GREP/output/mocov2_t2_5std.ICD.txt"

# Filter Threshold
P_VALUE_THRESHOLD = 0.05

# Visual Settings
FONT_SIZE = 16
OPACITY = 0.4  # Light pastel colors

# Colors (ICD Chapters) - Rotated so red colors go to F and G
chapter_colors = {
    'A': f'rgba(255, 195, 0, {OPACITY})',   # Yellow (was E)
    'B': f'rgba(218, 247, 166, {OPACITY})', # Light Green (was F)
    'C': f'rgba(51, 255, 87, {OPACITY})',   # Green (was G)
    'D': f'rgba(51, 255, 189, {OPACITY})',  # Teal (was H)
    'E': f'rgba(51, 128, 255, {OPACITY})',  # Blue (was I)
    'F': f'rgba(199, 0, 57, {OPACITY})',    # Red (was B) - RED!
    'G': f'rgba(255, 87, 51, {OPACITY})',   # Orange (was A) - RED!
    'H': f'rgba(93, 51, 255, {OPACITY})',   # Indigo (was J)
    'I': f'rgba(131, 51, 255, {OPACITY})',  # Violet (was K)
    'J': f'rgba(51, 128, 255, {OPACITY})',  # Blue
    'K': f'rgba(255, 51, 246, {OPACITY})',  # Pink (was M)
    'L': f'rgba(255, 51, 134, {OPACITY})',  # Hot Pink (was N)
    'M': f'rgba(144, 12, 63, {OPACITY})',   # Purple/Red (was C)
    'N': f'rgba(88, 24, 69, {OPACITY})',    # Dark Purple (was D)
}
fallback_color = f'rgba(136, 136, 136, {OPACITY})'

# Legend Names
icd_legend_map = {
    'A': 'Infectious (A)', 'B': 'Infectious (B)', 'C': 'Neoplasms',
    'D': 'Blood/Immune', 'E': 'Endocrine', 'F': 'Mental',
    'G': 'Nervous System', 'H': 'Eye/Ear', 'I': 'Circulatory',
    'J': 'Respiratory', 'K': 'Digestive', 'L': 'Skin',
    'M': 'Musculoskeletal', 'N': 'Genitourinary'
}

# --- 2. LOAD AND FILTER DATA ---
df = pd.read_csv(input_file, sep="\t")
df.columns = [c.lstrip("#") for c in df.columns]

# Clean P-Value column: Coerce errors to NaN (handles garbage text headers)
df['FisherExactP'] = pd.to_numeric(df['FisherExactP'], errors='coerce')

# *** PRIMARY FILTER: Keep only significant rows ***
df_sig = df[df['FisherExactP'] < P_VALUE_THRESHOLD].copy()

print(f"Original rows: {len(df)}. Rows with P < {P_VALUE_THRESHOLD}: {len(df_sig)}")

# --- 3. PARSE PATHS ---
flat_data = []

def clean_icd(val):
    if pd.isna(val): return None
    return val.split()[0] # Just the code, e.g., "C00-C14"

for _, row in df_sig.iterrows():
    raw_disease = row.get("Group")
    drug_field = row.get("TargetGene:DrugNames")

    if pd.isna(drug_field): continue

    disease_code = clean_icd(raw_disease)
    if not disease_code: continue
    
    # Determine Chapter
    chapter = disease_code[0] if disease_code[0].isalpha() else 'Other'

    # Parse GENE:drug,drug
    for block in str(drug_field).split(";"):
        if ":" not in block: continue
        gene, drugs_str = block.split(":", 1)
        gene = gene.strip()
        
        if not gene: continue

        for drug in drugs_str.split(","):
            drug = drug.strip()
            if drug:
                flat_data.append({
                    'gene': gene, 
                    'drug': drug, 
                    'disease': disease_code, 
                    'chapter': chapter
                })

# DataFrame of all valid connections
df_paths = pd.DataFrame(flat_data)

# Calculate statistics
num_sig_associations = len(df_sig)
num_unique_combinations = df_paths.drop_duplicates(['gene', 'drug', 'disease']).shape[0]

# Calculate unique counts
num_unique_genes = df_paths['gene'].nunique()
num_unique_drugs = df_paths['drug'].nunique()
num_unique_diseases = df_paths['disease'].nunique()

print(f"Number of significant associations: {num_sig_associations}")
print(f"Number of unique drug-gene-disease combinations: {num_unique_combinations}")
print(f"Number of unique genes: {num_unique_genes}")
print(f"Number of unique drugs: {num_unique_drugs}")
print(f"Number of unique diseases: {num_unique_diseases}")

if df_paths.empty:
    print("No data found passing the P-value filter.")
    exit()

# --- 4. PREPARE SANKEY DATA ---
# Drugs to highlight in red
highlight_drugs = {
    'amlodipine', 'asp-3258', 'butalbital', 'cilomilast', 'clobazam', 'clonazepam',
    'compound 1', 'diclofenac', 'dipyridamole', 'lorazepam', 'meprobamate', 'midazolam',
    'nbqx', 'nicardipine', 'nifedipine', 'nitroprusside', 'pentobarbital', 'propofol',
    'prostratin', 'sevoflurane', 'tas-203', 'th-9229', 'topiramate', 'trimebutine',
    'verapamil'
}

# Unique Lists
top_drugs2 = df_paths['drug'].value_counts().head(30).index
df_paths = df_paths[df_paths['drug'].isin(top_drugs2)].copy()

genes = sorted(df_paths['gene'].unique())
drugs = sorted(df_paths['drug'].unique())
diseases = sorted(df_paths['disease'].unique())

all_labels = genes + drugs + diseases
node_map = {label: i for i, label in enumerate(all_labels)}

# Create node colors: red for highlighted drugs, black for others
node_colors = []
for label in all_labels:
    if label.lower() in highlight_drugs:
        node_colors.append('red')
    else:
        node_colors.append('black')

# Create labels with red text for highlighted drugs
formatted_labels = []
for label in all_labels:
    if label.lower() in highlight_drugs:
        formatted_labels.append(f'<span style="color:red">{label}</span>')
    else:
        formatted_labels.append(label)

sources = []
targets = []
values = []
link_colors = []

# Path 1: Gene -> Drug
# Grouping maintains flow volume
gd_group = df_paths.groupby(['gene', 'drug', 'chapter']).size().reset_index(name='flow')
for _, row in gd_group.iterrows():
    sources.append(node_map[row['gene']])
    targets.append(node_map[row['drug']])
    values.append(row['flow'])
    link_colors.append(chapter_colors.get(row['chapter'], fallback_color))

# Path 2: Drug -> Disease
dd_group = df_paths.groupby(['drug', 'disease', 'chapter']).size().reset_index(name='flow')
for _, row in dd_group.iterrows():
    sources.append(node_map[row['drug']])
    targets.append(node_map[row['disease']])
    values.append(row['flow'])
    link_colors.append(chapter_colors.get(row['chapter'], fallback_color))

# --- 5. DYNAMIC HEIGHT ---
# Calculate height based on the busiest column so text doesn't overlap
max_nodes = max(len(genes), len(drugs), len(diseases))
dynamic_height = max(1000, max_nodes * 25) # 25px per node minimum

# --- 6. PLOT ---
fig = go.Figure()

# Sankey Trace
fig.add_trace(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=formatted_labels,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
))

# Legend Trace (Invisible Markers)
present_chapters = sorted(df_paths['chapter'].unique())
for chap in present_chapters:
    color_rgba = chapter_colors.get(chap, fallback_color)
    color_solid = color_rgba.replace(f", {OPACITY})", ", 1.0)")
    label_name = icd_legend_map.get(chap, f"Chapter {chap}")
    
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color=color_solid),
        name=label_name
    ))

# Layout: No Axis, Transparent Background
fig.update_layout(
    title_text=f"Significant Interactions (P < {P_VALUE_THRESHOLD})",
    font=dict(size=FONT_SIZE),
    height=dynamic_height,
    width=1600,
    showlegend=True,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

fig.show()

Original rows: 221. Rows with P < 0.05: 32
Number of significant associations: 32
Number of unique drug-gene-disease combinations: 1001
Number of unique genes: 75
Number of unique drugs: 370
Number of unique diseases: 32


In [23]:
set(top_drugs2)&set(top_drugs)

{'amlodipine',
 'asp-3258',
 'butalbital',
 'cilomilast',
 'clobazam',
 'clonazepam',
 'compound 1',
 'diclofenac',
 'dipyridamole',
 'lorazepam',
 'meprobamate',
 'midazolam',
 'nbqx',
 'nicardipine',
 'nifedipine',
 'nitroprusside',
 'pentobarbital',
 'propofol',
 'prostratin',
 'sevoflurane',
 'tas-203',
 'th-9229',
 'topiramate',
 'trimebutine',
 'verapamil'}